In [ ]:
__copyright__ = "Reiner Lemoine Institut gGmbH"
__license__ = "GNU Affero General Public License Version 3 (AGPL-3.0)"
__url__ = "https://github.com/openego/eDisGo/blob/master/LICENSE"

# Grid reinforcement along an expansion path

Grid expansion studies usually reinforce today's grid once for a single target year. In practice, a grid operator builds in stages: the grid is reinforced for 2030, and when more heat pumps and charging points arrive by 2035, the grid of 2030 is reinforced again.

This example shows how to calculate such an **expansion path** with `EDisGo.reinforce_expansion_path` and how to compare it with reinforcing the grid directly for the target year:

1. Load an example grid and reinforce today's state.
2. Describe the growth of heat pumps and home charging points for 2030, 2035 and 2040.
3. Reinforce the grid stage by stage and look at the costs per stage.
4. Compare the expansion path with building for each target year at once.
5. See how the comparison depends on the discount rate.

The example uses the small ding0 test grid shipped with eDisGo and needs no database access. It runs in about a minute.

### Import packages

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from edisgo import EDisGo
from edisgo.tools.logger import setup_logger

In [ ]:
# only show warnings and errors, reinforcement logs a lot on level info
setup_logger(
    loggers=[
        {"name": "root", "file_level": None, "stream_level": "warning"},
        {"name": "edisgo", "file_level": None, "stream_level": "warning"},
    ]
)

## Grid and today's state

We use the ding0 test grid from the eDisGo test data and a worst-case analysis (heavy load and reverse power flow).

The test grid already has overloaded components today. If we started the expansion path directly, the costs of the first stage would mostly be the costs of fixing today's problems. So we reinforce today's state first. The expansion path then only shows the costs caused by the growth.

In [ ]:
ding0_grid = os.path.join("..", "tests", "data", "ding0_test_network_1")

edisgo = EDisGo(ding0_grid=ding0_grid)
edisgo.set_time_series_worst_case_analysis()
edisgo.reinforce()

costs_today = edisgo.results.grid_expansion_costs.total_costs.sum()
print(f"Reinforcement of today's state: {costs_today:.1f} kEUR")

## Scenario: heat pumps and home charging points

Every stage is a function that takes the `EDisGo` object and applies the growth up to its year. The stages are applied **one after another to the same grid**, so the function of 2035 sees the grid of 2030 including everything added in 2030.

In this scenario, a growing share of the residential buildings gets a heat pump (8 kW electrical, including the heating rod) and an 11 kW home charging point. The order in which buildings get them is drawn once, so that a building with a heat pump in 2030 still has it in 2035.

In [ ]:
# share of residential buildings with heat pump and with home charging point
shares = pd.DataFrame(
    {"heat_pump": [0.3, 0.7, 1.0], "charging_point": [0.3, 0.6, 1.0]},
    index=pd.Index([2030, 2035, 2040], name="year"),
)
shares

In [ ]:
residential_buses = edisgo.topology.loads_df.query("sector == 'residential'").bus
# fixed order in which buildings get a heat pump or charging point
building_order = np.random.default_rng(42).permutation(residential_buses.to_numpy())


def add_up_to_share(edisgo_obj, comp_type, share, p_set, sector):
    """Adds components of the given type until `share` of the buildings have one."""
    loads = edisgo_obj.topology.loads_df
    buses_with_comp = set(loads[loads.type == comp_type].bus)
    for bus in building_order[: round(share * len(building_order))]:
        if bus not in buses_with_comp:
            edisgo_obj.add_component(
                "load", bus=bus, p_set=p_set, type=comp_type, sector=sector
            )


def stage(year):
    """Returns the function that applies the growth up to `year`."""

    def apply_stage(edisgo_obj):
        add_up_to_share(
            edisgo_obj,
            "heat_pump",
            shares.at[year, "heat_pump"],
            0.008,
            "individual_heating",
        )
        add_up_to_share(
            edisgo_obj,
            "charging_point",
            shares.at[year, "charging_point"],
            0.011,
            "home",
        )
        # the stage has to leave time series for all components
        edisgo_obj.set_time_series_worst_case_analysis()

    return apply_stage


stages = {year: stage(year) for year in shares.index}

Because the function only adds what is missing up to the share of its year, it works both on the expansion path and when all stages are applied at once to today's grid (which `greenfield=True` does, see below).

A function like `p_set *= 1.5` would be wrong here: it would grow the load by 1.5 in *every* stage that runs after it.

## Reinforce along the expansion path

`reinforce_expansion_path` processes the stages in the order of their years. For each stage, it applies the stage function, reinforces the grid with `EDisGo.reinforce` and calculates the costs of **the measures of this stage only**. The costs are discounted to `base_year`.

With `greenfield=True`, today's grid is additionally reinforced directly for the scenario of each year. This needs one more reinforcement per stage.

All other keyword arguments are passed on to `EDisGo.reinforce`, e.g. `reduced_analysis=True`.

In [ ]:
results = edisgo.reinforce_expansion_path(
    stages,
    base_year=2026,
    discount_rate=0.05,
    greenfield=True,
)
results.summary.round(1)

All costs are in kEUR:

| column | meaning |
|---|---|
| `costs` | costs of the measures taken in this stage |
| `costs_mv`, `costs_lv` | share in the MV grid (incl. HV/MV transformers) and in the LV grid (incl. MV/LV transformers) |
| `present_value` | `costs` discounted to `base_year` |
| `cumulative_costs`, `cumulative_present_value` | sums up to this stage |
| `greenfield_costs` | costs of reinforcing today's grid directly for the scenario of this year |

After the call, `edisgo` contains the grid reinforced for 2040.

In [ ]:
# colors: categorical slots 1 and 2 of the eDisGo example palette
PATH_COLOR, GREENFIELD_COLOR = "#2a78d6", "#eb6834"
TEXT_COLOR, MUTED_COLOR = "#0b0b0b", "#52514e"


def style_axes(ax, ylabel):
    ax.set_ylabel(ylabel, color=MUTED_COLOR)
    ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color("#c3c2b7")
    ax.tick_params(colors=MUTED_COLOR)


def label_line_end(ax, x, y, text, color):
    ax.annotate(
        text,
        (x, y),
        xytext=(8, 0),
        textcoords="offset points",
        va="center",
        color=TEXT_COLOR,
        fontsize=10,
    )
    ax.plot(
        x, y, "o", color=color, markersize=8, markeredgecolor="white", markeredgewidth=2
    )


summary = results.summary
years = summary.index

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    years,
    summary.cumulative_costs,
    color=PATH_COLOR,
    linewidth=2,
    marker="o",
    markersize=8,
    markeredgecolor="white",
    markeredgewidth=2,
    label="Expansion path (sum of all stages so far)",
)
ax.plot(
    years,
    summary.greenfield_costs,
    color=GREENFIELD_COLOR,
    linewidth=2,
    marker="o",
    markersize=8,
    markeredgecolor="white",
    markeredgewidth=2,
    label="Direct reinforcement for this year",
)
label_line_end(
    ax,
    years[-1],
    summary.cumulative_costs.iloc[-1],
    f"{summary.cumulative_costs.iloc[-1]:.0f} kEUR",
    PATH_COLOR,
)
label_line_end(
    ax,
    years[-1],
    summary.greenfield_costs.iloc[-1],
    f"{summary.greenfield_costs.iloc[-1]:.0f} kEUR",
    GREENFIELD_COLOR,
)
ax.set_xticks(years)
ax.set_xlim(years[0] - 1, years[-1] + 3)
ax.set_ylim(bottom=0)
style_axes(ax, "Grid expansion costs (kEUR, nominal)")
ax.set_title(
    "Building in stages vs. building for the target year at once",
    loc="left",
    color=TEXT_COLOR,
)
ax.legend(frameon=False, loc="upper left")
plt.show()

## What is built in which stage?

`results.grid_expansion_costs` and `results.equipment_changes` contain the measures of every stage with an additional column `year`.

In [ ]:
results.grid_expansion_costs.groupby(["year", "voltage_level"]).total_costs.agg(
    number_of_measures="count", costs="sum"
).round(1)

In [ ]:
changes = results.equipment_changes
changes[changes.equipment.str.contains("transformer")][["equipment", "change", "year"]]

## Why is the path more expensive here?

The MV/LV stations 8 and 9 get one more transformer in **every** stage. Each stage only needs a bit more capacity, and `reinforce` covers that by adding another transformer of the existing type (100 kVA) in parallel. Reinforcing directly for 2040 instead replaces the 100 kVA transformer with a single 630 kVA standard transformer.

In the default cost configuration every MV/LV transformer costs the same, regardless of its size, so the path pays for three transformers where the direct reinforcement pays for one. The same effect occurs with lines: a cable laid in a later stage needs its own trench, so the earthworks are paid again.

In [ ]:
transformers = edisgo.topology.transformers_df
transformers[transformers.bus0.str.contains("LVGrid_8|LVGrid_9")][
    ["bus0", "type_info", "s_nom"]
]

## Does postponing pay off?

Building in stages costs more in total, but part of the money is spent later. Whether this pays off depends on the discount rate. The costs per stage do not depend on it, so we can recalculate the present values for any rate without reinforcing again.

For a fair comparison, the direct reinforcement for 2040 is assumed to be built in the first stage year, 2030.

In [ ]:
base_year = 2026
rates = np.linspace(0, 0.15, 151)
path_pv = [
    (summary.costs / (1 + r) ** (summary.index - base_year)).sum() for r in rates
]
greenfield_pv = [
    summary.greenfield_costs.iloc[-1] / (1 + r) ** (years[0] - base_year) for r in rates
]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    rates * 100,
    path_pv,
    color=PATH_COLOR,
    linewidth=2,
    label="Expansion path 2030 / 2035 / 2040",
)
ax.plot(
    rates * 100,
    greenfield_pv,
    color=GREENFIELD_COLOR,
    linewidth=2,
    label="Direct reinforcement for 2040, built in 2030",
)
ax.set_xlim(0, 15)
ax.set_ylim(bottom=0)
ax.set_xlabel("Discount rate (%)", color=MUTED_COLOR)
style_axes(ax, f"Present value in {base_year} (kEUR)")
ax.set_title("Present value of both strategies", loc="left", color=TEXT_COLOR)
ax.legend(frameon=False, loc="upper right")
plt.show()

cheaper = np.array(path_pv) < np.array(greenfield_pv)
if cheaper.any():
    print(
        f"The expansion path is cheaper from a discount rate of "
        f"{rates[cheaper.argmax()] * 100:.1f} % on."
    )
else:
    print("The direct reinforcement is cheaper for all discount rates up to 15 %.")

## Using it with your own data

The stage functions can do anything that brings the grid to the state of their year, e.g. import a generator scenario, add heat pumps from your own data or scale loads. Things to keep in mind:

* The stage function has to leave time series for all components. In a worst-case analysis, call `set_time_series_worst_case_analysis()` at its end. With time series, set the time series of new components.
* The stage functions are applied cumulatively. Write them so that applying the stages one after another gives the state of each year.
* `edisgo` is changed in place. Use `copy.deepcopy(edisgo)` if you need today's grid afterwards.
* Connection lines of new components count as costs of their stage. Pass `without_generator_import=True` to exclude them, as in `EDisGo.reinforce`.

Not included yet: residual values of equipment at the end of the period, replacement of old equipment, and transformer costs that depend on the transformer size.